In [ ]:
import os

os.chdir("..")
os.getcwd()

'/home/szczuru/si-tone'

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchaudio
import numpy as np
import json
import os
from pathlib import Path

from models.confromer.conformer import Conformer
from utils.tokenizer import Tokenizer
from data.aishell_dataset import AishellDataset
from data.collate import collate_fn

# config stuff...
checkpoint_path = "checkpoints/conformer_epoch_6.pt"
config_path = "checkpoints/config.json"

dev = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
with open(config_path) as f:
    cfg = json.load(f)
cfg

{'data_path': 'data_aishell',
 'vocab_path': 'utils/vocab.json',
 'checkpoint': 'checkpoints/conformer.pt',
 'sample_rate': 16000,
 'n_mels': 80,
 'batch_size': 32,
 'num_epochs': 100,
 'lr': 0.0005,
 'warmup_steps': 4000,
 'd_model': 144,
 'num_heads': 4,
 'ffn_dim': 576,
 'num_layers': 16,
 'conv_kernel': 31,
 'dropout': 0.1}

In [ ]:
aishell_ds = AishellDataset(cfg['data_path'], sample_rate=cfg['sample_rate'], n_mels=cfg['n_mels'])
aishell_dl = DataLoader(
    dataset=aishell_ds,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
)

print(f'Dataset size: {len(aishell_ds)}')

tokenizer = Tokenizer()
tokenizer.load_vocab(cfg['vocab_path'])

model = Conformer(
    input_dim=cfg['n_mels'],
    d_model=cfg['d_model'],
    num_heads=cfg['num_heads'],
    ffn_dim=cfg['ffn_dim'],
    num_layers=cfg['num_layers'],
    conv_kernel=cfg['conv_kernel'],
    dropout=cfg['dropout'],
    vocab_size=len(tokenizer),
).to(dev)

Dataset size: 69843


In [ ]:
ckpt = torch.load(checkpoint_path, weights_only=False, map_location='cpu')
model.load_state_dict(ckpt['model'])

<All keys matched successfully>

In [ ]:
model.eval()

Conformer(
  (encoder): Encoder(
    (input_proj): Linear(in_features=80, out_features=144, bias=True)
    (pos_encoding): RelPosEncoding()
    (layers): ModuleList(
      (0-15): 16 x ConformerBlock(
        (ffn1): FeedForwardModule(
          (seq): Sequential(
            (0): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=144, out_features=576, bias=True)
            (2): SiLU()
            (3): Dropout(p=0.1, inplace=False)
            (4): Linear(in_features=576, out_features=144, bias=True)
            (5): Dropout(p=0.1, inplace=False)
          )
        )
        (self_attn): SelfAttentionModule(
          (norm): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
          (attn): RelativeMultiHeadAttention(
            (q_proj): Linear(in_features=144, out_features=144, bias=True)
            (k_proj): Linear(in_features=144, out_features=144, bias=True)
            (v_proj): Linear(in_features=144, out_features=144, bias=True)

In [ ]:
num_samples = 20
all_mels, all_mel_lengths, all_targets, all_target_lengths = next(iter(
    DataLoader(aishell_ds, batch_size=num_samples, shuffle=True, collate_fn=collate_fn)
))
all_mels = all_mels.to(dev)
all_mel_lengths = all_mel_lengths.to(dev)

with torch.no_grad():
    log_probs = model(all_mels, all_mel_lengths)  # (B, T, vocab)

log_probs_t = log_probs.transpose(0, 1)  # (T, B, vocab)
pred_ids = log_probs_t.argmax(dim=-1)    # (T, B)
blank_id = tokenizer.char_to_id['<blank>']

for b in range(num_samples):
    ids = pred_ids[:all_mel_lengths[b], b].tolist()
    collapsed = []
    prev = None
    for idx in ids:
        if idx != prev and idx != blank_id:
            collapsed.append(idx)
        prev = idx
    predicted = ''.join(tokenizer.id_to_char[i] for i in collapsed)
    target    = ''.join(tokenizer.id_to_char[i] for i in all_targets[b, :all_target_lengths[b]].tolist())
    print(f'[{b:02d}] target:    {target}')
    print(f'[{b:02d}] predicted: {predicted}')
    print()

print('raw ids (first 30):', pred_ids[:30, 0].tolist())
print('unique id w batchu:', pred_ids.unique().tolist())
print('log_probs min/max:', log_probs.min().item(), log_probs.max().item())

[00] target:    yāoqǐnglebāwèizhōngguóxuǎnshǒuchūzhàn
[00] predicted: yāoqǐngbāwèizhōngguóxuéshēngchūzhàn

[01] target:    zhèxiēwèntíyǐngxiǎnggōnggòngcǎigòushìchǎngdeguīfànyǔwánshàn
[01] predicted: zhèxiēwènjīyǐngxiǎnggōnggōngcǎibùshìchǎngdeguīfànyǔwánshàn

[02] target:    tóngbǐzēngzhǎngbǎifēnzhījiǔdiǎnèryī
[02] predicted: tóngbǐzēngzhǎngbǎifēnzhījiǔdiǎnèryì

[03] target:    zhúbùjiànshèguīfàntǒngyīdezhàiquànshìchǎng
[03] predicted: zhúbùjiànshèguīfàntǒnglìdezhàiquànshìchǎng

[04] target:    huìrànghǎochùzhēnzhèngluòdàolǎobǎixìngshēnshàng
[04] predicted: huìrànghǎochūzhēnliàngrèdelāwěijìnshēnshàng

[05] target:    zhōngguónánziyǒngjūnqiángshìjuéqǐnǚzǐhùnhéjiēlìduóguànzhǎnlùxīwàng
[05] predicted: zhōngguónánziyǒngjūnchángshìjuéqǐyùcǐhuánhéjiēlǐduóguǎnzhǎnrùxīwǎng

[06] target:    guóqīngdùqīngqīngdǎdiàojiéhéyǐqīsìqiǎngzhànxiānjī
[06] predicted: luòqiánzhīqiánquēfǎdiàojiéhéyǐqīsìqiángzhànqiánjī

[07] target:    dànshēnwèishǒuqiúduìyuándetāmenquèbèigǎnqīngsōng
[07] predicted: dànxiāowèi

In [ ]:
from utils.to_pinyin import to_pinyin

model.eval()
mels, mel_lengths, targets, target_lengths = next(iter(aishell_dl))
mels = mels.to(dev)
mel_lengths = mel_lengths.to(dev)

with torch.no_grad():
    log_probs = model(mels, mel_lengths)  # (B, T, vocab)

log_probs_t = log_probs.transpose(0, 1)  # (T, B, vocab)
pred_ids = log_probs_t.argmax(dim=-1)    # (T, B)
blank_id = tokenizer.char_to_id['<blank>']

for b in range(len(mel_lengths)):
    ids = pred_ids[:mel_lengths[b], b].tolist()
    collapsed = []
    prev = None
    for idx in ids:
        if idx != prev and idx != blank_id:
            collapsed.append(idx)
        prev = idx
    predicted = tokenizer.decode(collapsed)
    target_ids = targets[b, :target_lengths[b]].tolist()
    target = tokenizer.decode(target_ids)
    print(f'[{b:02d}] predicted: {predicted}')
    print(f'[{b:02d}] target:    {target}')
    print()

[00] predicted: shì shí wén zuò de tuī chū cǐ xiàng cuò shì
[00] target:    shì shí wěn tuǒ dì tuī chū cǐ xiàng cuò shī

[01] predicted: kě chuān dài shè bèi de shì chǎng guī mù yě zài xū sù kuò dà
[01] target:    kě chuān dài shè bèi de shì chǎng guī mó yě zài xùn sù kuò dà

[02] predicted: zài zhōng gāo wǎng jiāo yì dìng míng míng xiǎn de shí jiān duàn
[02] target:    zài zhōng gāng wǎng jiāo yì bìng wú míng xiǎn de shí jiān duàn

[03] predicted: yǔ bǐ kē liù bǎi qí shí qī sān de chéng jì bèi shuō dān dà xué lù qù
[03] target:    yǐ lǐ kē liù bǎi qī shí qī fēn de chéng jì bèi fù dàn dà xué lù qǔ

